# GroundingDINO + SAM 2 자동 주석 초안과 CVAT 검수

이 노트북은 AIHub 후보 이미지 500장에 접시 전체(`plate_full`)와 보이는 음식(`food_visible`) 초안을 만들고, CVAT에서 **검수만** 하도록 준비합니다. GPU 런타임을 사용하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'프로젝트 경로를 확인하세요: {PROJECT_ROOT}'
%cd {PROJECT_ROOT}

In [ ]:
# 기존 코랩 PyTorch를 유지하고 프로젝트의 호환 의존성만 보완합니다.
!pip install --prefer-binary --upgrade-strategy only-if-needed -r requirements-colab.txt
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'

In [ ]:
# 자동 주석보다 먼저, 정확히 같은 500장과 작업 목록 CSV를 준비합니다.
from pathlib import Path
WORK_ROOT = Path('data/training/plate_segmentation')
manifest = WORK_ROOT / 'plate_annotation_manifest.csv'
images_dir = WORK_ROOT / 'cvat_images'
RESET_PLATE_WORKSPACE = False  # 이전 53장 등 불완전 작업 폴더를 새 500장으로 교체할 때만 True
if not manifest.exists():
    if images_dir.exists() and any(images_dir.iterdir()) and not RESET_PLATE_WORKSPACE:
        raise RuntimeError(
            'cvat_images는 있지만 plate_annotation_manifest.csv가 없습니다. '
            '기존 결과를 백업한 뒤 이 셀의 RESET_PLATE_WORKSPACE=True로 바꾸고 다시 실행하세요.'
        )
    reset_arg = '--reset-output' if RESET_PLATE_WORKSPACE else ''
    !python -m scripts.prepare_plate_annotation_manifest --sample-size 500 {reset_arg}
image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
assert manifest.is_file(), f'작업 목록 생성 실패: {manifest}'
assert len(image_files) == 500, f'입력 이미지는 500장이어야 합니다. 현재: {len(image_files)}장'
print('작업 목록과 입력 이미지 확인 완료:', len(image_files), '장')

In [ ]:
# 첫 실행은 GroundingDINO와 SAM 2.1 Small 가중치를 내려받으므로 시간이 걸립니다.
# 원본 GroundingDINO/SAM2 저장소를 별도 설치하지 마세요. 현재 프로젝트 어댑터를 사용합니다.
!python -m scripts.generate_plate_segmentation_drafts --overwrite

In [ ]:
import json
from IPython.display import Image, display
summary = json.loads(Path('data/training/plate_segmentation/auto_annotations/draft_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
previews = sorted(Path(summary['preview_dir']).glob('*.jpg'))
if previews:
    display(Image(filename=str(previews[0])))
print('CVAT 업로드 파일:', summary['coco_json'])

## CVAT 검수 뒤

`cvat_images`를 CVAT Task에 올린 뒤 `instances_draft.json`을 COCO Instances 형식으로 가져옵니다. 접시 외곽, 보이는 음식, 누락만 수정하고 COCO Instances로 내보냅니다. 내보낸 파일을 `data/training/plate_segmentation/annotations/instances_reviewed.json`에 둡니다. 자세한 초보자 안내는 `docs/GROUNDED_SAM2_CVAT_REVIEW.md`를 참고하세요.

In [ ]:
# CVAT 검수본을 Drive에 둔 뒤 실행합니다.
reviewed = Path('data/training/plate_segmentation/annotations/instances_reviewed.json')
assert reviewed.is_file(), f'검수 COCO 파일이 없습니다: {reviewed}'
!python -m scripts.mark_plate_annotation_reviewed --coco-json {reviewed}
!python -m scripts.prepare_plate_segmentation_dataset --coco-json {reviewed}
!python -m scripts.train_yolo11n_plate_segmenter --epochs 100 --imgsz 1024
!python -m scripts.evaluate_yolo11n_plate_segmenter --weights runs/plate_segmenter/yolo11n_plate_seg_v1/weights/best.pt